In [0]:
%sql

-- PIPELINE: BRONZE -> SILVER
MERGE INTO catalog_ventas.silver.ventas_clean AS target
USING (

  WITH datos_limpios AS (
      SELECT 
        venta,
        articulo,
        -- Limpieza basica
        NULLIF(
          TRIM(
            regexp_replace(
                regexp_replace(
                  TRANSLATE(LOWER(descrip), 'áéíóúüñ', 'aeiouun'),
                    '[^a-z0-9 ]', 
                      ' '
                    ),
                    ' +',
                    ' '
                    )
                    ),
                    ''
                  ) AS descrip_clean,
                
      TRY_CAST(precio AS DECIMAL(15,2)) AS precio,
      cant,
      TRY_CAST(total AS DECIMAL(15,2)) AS total,

      LOWER(TRIM(usulogin)) AS usulogin,
      LOWER(TRIM(clinombre)) AS clinombre,
    
      vtaoperacion,

      LOWER(TRIM(vtaestado)) AS vtaestado,
      TRY_TO_TIMESTAMP(vtafecha, 'd/M/yyyy HH:mm') AS vtafecha,
      condvtapos,
      LOWER(TRIM(delivery)) AS delivery,

      comprobante
   
      FROM catalog_ventas.raw.ventas_heladeria_bronze

  ),
  datos_normaliazdos AS (
  SELECT
    venta,
    articulo,

    -- Estandarización + eliminación de "envios a domicilio"
    CASE 
        WHEN descrip_clean LIKE '%1 2 kilo%' THEN '1/2kilo'
        WHEN descrip_clean LIKE'%1 kilo%' OR descrip_clean LIKE '%1 kg%' THEN '1kilo'
        WHEN descrip_clean LIKE '%1/4%' THEN '1/4kilo'
        WHEN descrip_clean LIKE '%sabores%' THEN '1/4kilo'
        WHEN descrip_clean LIKE 'torta grido rellen' THEN 'torta especial relleno'
        ELSE descrip_clean
    END AS descrip,
    precio,
    cant,
    total,
    usulogin,
    
    -- Normalizar clientes
    CASE 
        WHEN vtaoperacion = 'VL' OR vtaoperacion='CL' THEN 'socio'
        WHEN vtaoperacion = 'VF' THEN 'no_socio'
        WHEN vtaoperacion = 'VG' THEN 'gastronomico'
    END AS cliente,
    
    vtaestado,
    vtafecha,

    --Normalizar turno (mañana/noche)
    CASE    
        WHEN HOUR(vtafecha) BETWEEN 10 AND 18 THEN 'mañana'
        ELSE 'noche' 
    END AS  turno,

    --Estandarizacion de medios de pago
    CASE 
        WHEN condvtapos='TD' THEN 'tarjeta'
        WHEN condvtapos='TC' THEN 'qr'
        WHEN condvtapos='EF' THEN 'efectivo'
        WHEN condvtapos='MU' THEN 'multiple_opciones' 
        WHEN condvtapos='XX' THEN 'cancelado'
        ELSE '0'
    END AS medio_pago,

   --Normalizar valores (si/no) 
   CASE 
        WHEN delivery IN ('si','1','true') THEN 1
        ELSE 0
    END AS delivery,

    --Normalizar valores nulls de comprobante
    CASE WHEN comprobante IS NULL THEN 'sin_comprobante' ELSE comprobante END AS comprobante,

    --Normalizar columna descrip
    CASE
        WHEN LOWER(descrip) LIKE '%torta%' THEN 'tortas'

        WHEN LOWER(descrip) LIKE '%pizza%' 
          OR LOWER(descrip) LIKE '%pechuguitas%'
          OR LOWER(descrip) LIKE '%frizzio%'  
          OR LOWER(descrip) LIKE '%empanada%' THEN 'congelados'

        WHEN LOWER(descrip) LIKE '%tent%' THEN 'helado_1l'
        
        WHEN LOWER(descrip) LIKE '%1 kg%' 
          OR LOWER(descrip) LIKE '%1/2%'
          OR LOWER(descrip) LIKE '%1/4%'
          OR LOWER(descrip) LIKE '%kilo%'
          OR LOWER(descrip) LIKE '%toy 2 boch%'
          OR LOWER(descrip) LIKE '%boch%' THEN 'granel'

        WHEN LOWER(descrip) LIKE '%familiar%' THEN 'familiar'

        WHEN LOWER(descrip) LIKE '%pal%' 
          OR LOWER(descrip) LIKE '%cremoso%'
          OR LOWER(descrip) LIKE '%frutilla%'
          OR LOWER(descrip) LIKE '%lim%'
          OR LOWER(descrip) LIKE '%palito%' THEN 'palitos'

        WHEN LOWER(descrip) LIKE '%bombon%' 
          OR LOWER(descrip) LIKE '%alfajor%'
          OR LOWER(descrip) LIKE 'b%' THEN 'bombones'

        WHEN LOWER(descrip) LIKE '%cucur%' 
          OR LOWER(descrip) LIKE '%tacita%' 
          OR LOWER(descrip) LIKE '%bocha%' 
          OR LOWER(descrip) LIKE '%vasito%' 
          OR LOWER(descrip) LIKE '%gigante%'
          OR LOWER(descrip) LIKE '%gridito%'
          OR LOWER(descrip) LIKE '%grido%'
          OR LOWER(descrip) LIKE '%cups%' THEN 'helado_bocha'

        WHEN LOWER(descrip) LIKE '%smoothie%' 
          OR LOWER(descrip) LIKE '%batido%' THEN 'bebidas'

        WHEN LOWER(descrip) LIKE '%yogurt%' THEN 'yogurt_helado'

        WHEN LOWER(descrip) LIKE '%veg%' THEN 'vegano'

        WHEN LOWER(descrip) LIKE '%doble c%' THEN 'bocaditos'

         WHEN LOWER(descrip)  LIKE '%hel dul%' 
            OR LOWER(descrip) LIKE '%helado dulzura%' THEN 'helado_sin_azucar'

        WHEN LOWER(descrip) LIKE '%almendrado%'
          OR LOWER(descrip) LIKE '%delicia%'
          OR LOWER(descrip) LIKE '%crocantino%' 
          OR LOWER(descrip) LIKE '%casatta%' THEN 'postres_helados'

        WHEN LOWER(descrip) LIKE '%sundae%'
          OR LOWER(descrip) LIKE '%go%'
          OR LOWER(descrip) LIKE '%shot%'
          OR LOWER(descrip) LIKE '%dulce de leche%'
          OR LOWER(descrip) LIKE '%mermelad%'
          OR LOWER(descrip) LIKE '%shot%'
          OR LOWER(descrip) LIKE '%capelina%' THEN 'especiales'
        ELSE 'otros'
    END AS categoria
    
    FROM datos_limpios
 
  ),
  
  datos_duplicados AS (
      SELECT *,
             ROW_NUMBER() OVER (
                 PARTITION BY venta,articulo
                 ORDER BY vtafecha DESC
             ) AS rn,
              md5(
               concat_ws('||', venta, articulo, descrip, categoria, precio, cant, total, usulogin, cliente, vtaestado, vtafecha, turno, medio_pago, delivery)
           ) AS hash_md5
      FROM datos_normaliazdos
  )
  SELECT
      venta,
      articulo,
      descrip,
      categoria,
      precio,
      cant,
      total,
      usulogin,
      cliente,
      vtaestado,
      vtafecha,
      turno,
      medio_pago,
      delivery,
      hash_md5,
      'bronze_EDA' AS _source_table,
      CURRENT_TIMESTAMP AS _processing_timestamp
  FROM datos_duplicados
  WHERE rn = 1 
        AND precio IS NOT NULL
        AND precio > 0
        AND precio <> 0.01
        AND total > 0
        AND total <> 0.01
        AND articulo IS NOT NULL AND articulo <> 0
        AND descrip IS NOT NULL

) AS source

ON target.venta = source.venta
  AND target.articulo = source.articulo

WHEN MATCHED AND target.hash_md5 <> source.hash_md5 THEN UPDATE SET
  target.articulo              = source.articulo,
  target.descrip               = source.descrip,
  target.categoria             = source.categoria,
  target.precio                = source.precio,
  target.cant                  = source.cant,
  target.total                 = source.total,
  target.usulogin              = source.usulogin,
  target.cliente               = source.cliente,
  target.vtaestado             = source.vtaestado,
  target.vtafecha              = source.vtafecha,
  target.turno                 = source.turno,
  target.medio_pago            = source.medio_pago,           
  target.delivery              = source.delivery

WHEN NOT MATCHED THEN INSERT  (
  venta,
  articulo,
  descrip,
  categoria,
  precio,
  cant,
  total,
  usulogin,
  cliente,
  vtaestado,
  vtafecha,
  turno,
  medio_pago,
  delivery,
  _source_table,
  _processing_timestamp,
  hash_md5
)
VALUES(
  source.venta,
  source.articulo,
  source.descrip,
  source.categoria,
  source.precio,
  source.cant,
  source.total,
  source.usulogin,
  source.cliente,
  source.vtaestado,
  source.vtafecha,
  source.turno,
  source.medio_pago,
  source.delivery,
  source._source_table,
  CURRENT_TIMESTAMP(),
  source.hash_md5
)


